## LDA topic modeling


The utility of topic modeling methods is their capability to uncover unobserved variables—topics—which shape the meaning of textual documents. Modern-day scholars utilize topic modeling to uncover latent topics from a wide array of textual information—from shorter texts, such as twitter posts to longer texts, such as journal articles.



This notebook applies LDA modeling to an experimental dataset investigating participants' goal inferences. 


### Key python libraries:
- gensim (https://radimrehurek.com/gensim/)
- nltk (https://www.nltk.org)
- spacy (https://spacy.io)

### Helpful Links:
- https://medium.com/@lettier/how-does-lda-work-ill-explain-using-emoji-108abf40fa7d
- https://towardsdatascience.com/light-on-math-machine-learning-intuitive-guide-to-latent-dirichlet-allocation-437c81220158






## A Comprehensive Example:

The data represent textual participants’ responses of 812 participants of the questionnaire described in the paper "A Theory-Driven Computational Measure of the Goal Construct in Communication Science". Responses from a single participant are aggregated together in a single document, leaving a total of 812 documents combined in a single data set.

LDA assumes that a single document can contribute to multiple topics simultaneously; in other words, LDA explicitly models the actual distribution of words within each document. However, this assumption has drawbacks when modeling relatively shorter texts (e.g., twitter posts) because these documents may not contain enough meaningful words to model their distribution across topics. As a result, LDA models of longer texts produce more variance than with shorter texts—raising concerns when modeling open-ended responses.

The goal of the analysis is to investigate participants’ responses of the questionnaire.

##  Steps of the analysis:

#### 1. Preparing data for LDA
    a. Spell check
    b. Expand contractions
    c. Read the data 
    d. Check data integrity
    e. Delete missing values
#### 2. Text preprocessing
    a. Tokenization
    b. Lemmatization  
    c. Stop Word Removal
    d. Bigrams and Trigrams
    e. Exclude terms in > 99% and < 1% of documents
    f. Generate Corpus and Dictionary
#### 3. Model selection (Selecting the number of topics (k))
    a. Computing Model Perplexity
    b. Analyzing model results through pyLDAvis visualization
    c. Saving selected model results

In [46]:
## Load Required Libraries

#general
import numpy as np
import pandas as pd
import re
import pickle
from IPython.display import display

#setting up Jupyter notebook 
%matplotlib inline
pd.set_option('display.max_rows', 5000)
pd.set_option('display.max_columns', 5000)
pd.set_option('display.width', 10000)

#text preprocessing
import nltk
from nltk.corpus import stopwords

import spacy
from spacy.lang.en import English

from gensim.models import Phrases
from gensim.utils import simple_preprocess


#modeling
import gensim
from gensim.models.ldamodel import LdaModel


#plotting
import pyLDAvis
import pyLDAvis.gensim

### 1. Preparing data for LDA

Before you read in your data, you should manually run your textual data through a spell checker in order to take advantage of the semantic and syntactic context when selecting the proper correct spelling.

Similar to the spellchecker, we needed human coders to expand all English contractions (e.g., "don't" -> "do not"), to ensure accuracy.

After you read the data in, you should check its integrity to avoid unexpected and artificial errors and delete missing values (null values).

In [47]:
## File paths

#data files
file_location = './data/experimental_data.xlsx'

#stop words
stopwords_location = './data/stopwords.txt'

In [48]:
## Check to make sure the dataset looks correct
try:
    data = pd.read_excel(file_location, encoding='latin1')
    print("{} Rows.  {} Columns.".format(*data.shape))
except:
    print("Dataset could not be loaded. Is the dataset missing?")

813 Rows.  283 Columns.


In [49]:
## Spot checks
indices = [0,333,777]

samples = pd.DataFrame(data.loc[indices, :], columns = data.keys()).reset_index(drop = True)
print("Sample Tickets:")
display(samples)

Sample Tickets:


,uid,All_merged,GI_AM_merged,GI_Femstud_merged,GI_Author_merged,GI_Author_merged_oroginal,StartDate,EndDate,Status,IPAddress,Progress,Duration (in seconds),Finished,RecordedDate,ResponseId,RecipientLastName,RecipientFirstName,RecipientEmail,ExternalReference,LocationLatitude,LocationLongitude,DistributionChannel,UserLanguage,time_introv_First Click,time_introv_Last Click,time_introv_Page Submit,time_introv_Click Count,time_instr_First Click,time_instr_Last Click,time_instr_Page Submit,time_instr_Click Count,timer_01_First Click,timer_01_Last Click,timer_01_Page Submit,timer_01_Click Count,timer_02_First Click,timer_02_Last Click,timer_02_Page Submit,timer_02_Click Count,timer_03_First Click,timer_03_Last Click,timer_03_Page Submit,timer_03_Click Count,timer_04_First Click,timer_04_Last Click,timer_04_Page Submit,timer_04_Click Count,timer_05_First Click,timer_05_Last Click,timer_05_Page Submit,timer_05_Click Count,timer_06_First Click,timer_06_Last Click,timer_06_Page Submit,timer_06_Click Count,timer_07_First Click,timer_07_Last Click,timer_07_Page Submit,timer_07_Click Count,timer_08_First Click,timer_08_Last Click,timer_08_Page Submit,timer_08_Click Count,timer_09_First Click,timer_09_Last Click,timer_09_Page Submit,timer_09_Click Count,timer_10_First Click,timer_10_Last Click,timer_10_Page Submit,timer_10_Click Count,timer_11_First Click,timer_11_Last Click,timer_11_Page Submit,timer_11_Click Count,timer_12_First Click,timer_12_Last Click,timer_12_Page Submit,timer_12_Click Count,timer_13_First Click,timer_13_Last Click,timer_13_Page Submit,timer_13_Click Count,timer_14_First Click,timer_14_Last Click,timer_14_Page Submit,timer_14_Click Count,gi_AM_1,gi_AM_2,gi_AM_3,gi_AM_4,gi_AM_5,gi_AM_6,gi_AM_7,gi_AM_8,gi_AM_9,gi_AM_10,gi_femstud_1,gi_femstud_2,gi_femstud_3,gi_femstud_4,gi_femstud_5,gi_femstud_6,gi_femstud_7,gi_femstud_8,gi_femstud_9,gi_femstud_10,gi_author_1,gi_author_2,gi_author_3,gi_author_4,gi_author_5,gi_author_6,gi_author_7,gi_author_8,gi_author_9,gi_author_10,gi_author_1 gi_author_2 gi_author_3 gi_author_4 gi_author_5 gi_author_6 gi_author_7 gi_author_8 gi_author_9 gi_author_10,blame_femstud_1,blame_femstud_2,blame_femstud_3,blame_femstud_4,blame_femstud_5,blame_femstud_6,blame_AM_1,blame_AM_2,blame_AM_3,blame_AM_4,blame_AM_5,blame_AM_6,relative_blame1_1,relative_blame1_2,relative_blame2_1,relative_blame2_2,relative_blame2_3,rape1_1,rape1_2,rape1_3,rape1_4,rape1_5,rape1_6,rape1_7,rape1_8,rape1_9,rape1_10,Consent_before_1,Consent_before_2,Consent_before_3,Consent_before_4,Consent_before_5,Consent_before_6,Consent_before_7,Consent_before_8,Consent_before_9,Consent_before_10,Consent_before_11,Consent_before_12,Consent_after_1,Consent_after_2,Consent_after_3,Consent_after_4,Consent_after_5,Consent_after_6,Consent_after_7,Consent_after_8,Consent_after_9,Consent_after_10,Consent_after_11,Consent_after_12,REMV_1,REMV_2,REMV_3,REMV_4,REMV_5,REMV_6,REMV_7,REMV_8,REMP_1,REMP_2,REMP_3,REMP_4,REMP_5,REMP_6,REMP_7,REMP_8,status_1,status_2,status_3,status_4,status_5,status_6,status_7,status_8,status_9,status_10,status_11,status_12,status_13,status_14,status_15,status_16,position_recog,quote_recog,alcohol_morris,alcohol_femstudent,alcohol_present,intoxication_AM_1,intoxication_AM_2,intoxication_AM_3,intoxication_femstud_1,intoxication_femstud_2,intoxication_femstud_3,realism_1,realism_2,realism_3,realism_4,realism_5,realism_6,realism_7,realism_8,realism_9,worked_aggie,journalism_exp,partic_sex_assault,age,sex,race,race_7_TEXT,orientation,orientation_5_TEXT,RMA_1,RMA_2,RMA_3,RMA_4,RMA_5,RMA_6,RMA_7,RMA_8,RMA_9,RMA_10,RMA_11,RMA_12,RMA_13,RMA_14,RMA_15,RMA_16,RMA_17,RMA_18,RMA_19,RMA_20,RMA_21,RMA_22,RMA_23,RMA_24,RMA_25,RMA_26,RMA_27,RMA_28,RMA_29,RMA_30,RMA_31,RMA_32,RMA_33,RMA_34,RMA_35,RMA_36,RMA_37,RMA_38,RMA_39,RMA_40,RMA_41,RMA_42,RMA_43,RMA_44,RMA_45,alchohol,status1,status2,topic,gi_author_8 - Sentiment Polarity,gi_author_8 - Sentiment Score,gi_author_8 - Sentiment,gi_author_8 - Topics
0,1.0,NaN,NaN,NaN,Get readers Expos

In [50]:
## Check number of null values in each column of the full dataset
pd.DataFrame(data.isnull().sum(), columns=['Number of NULL values'])

,Number of NULL values
uid,1
All_merged,813
GI_AM_merged,813
GI_Femstud_merged,813
GI_Author_merged,0
GI_Author_merged_oroginal,0
StartDate,1
EndDate,1
Status,1
IPAddress,6


In [51]:
## Remove missing (null) values from the data

#finding null values in the full dataset
print("=============Full Dataset=============")
data['GI_Author_merged'] = data['GI_Author_merged']

print('Number of rows in GI_Author_merged:', len(data['GI_Author_merged']))
print("-------------------")
print("Null Values in GI_Author_merged: {}".format(data['GI_Author_merged'].isnull().sum()))

#removing null values from the full dataset
GI_Author_merged = data['GI_Author_merged']

print("After removing Null Values in Full Dataset")
print("Null Values in GI_Author_merged: {}".format(data['GI_Author_merged'].isnull().sum()))

=============Full Dataset=============
Number of rows in GI_Author_merged: 813
-------------------
Null Values in GI_Author_merged: 0
After removing Null Values in Full Dataset
Null Values in GI_Author_merged: 0


### 2. Text preprocessing
This step is needed to generate a ‘bag-of-words’ LDA model and it includes: text tokenization and lemmatization, removal of Stop Words and words that appear in > 99% and < 1% of documents, including bigrams and trigrams, and generating Corpus and Dictionary.

**Tokenization** involves converting the text to lowercase, removing special characters, and punctuation from the text. Also, we should be careful to remove alphanumerics, numbers, words that appear in the corpus less than twice and extra spaces.


**Lemmatization** is used reduces the size of the vocabulary in the model. It transforms words to their lemma (e.g., assaulted -> assault). So that the model can analyze several inflected forms of a word as a single word. Also, lemmatization using Spacy allows to select certain part of speech words (e.g., noun, adj, vb, adv).


**Stop Word Removal** often is an important step to have a better input for modeling. Stop words are very common words in a language (e.g. a, an, the etc.). Note: you can edit the stop words txt file to add additional words to filter out.  We recommend filtering out as few stop words as possible, as even commonly occurring words can offer meaningful information, especially when responses are terse. However, depending on the specific characteristics of the textual data, stop word removal may be necessary to minimize model noise.

**Bigrams and Trigrams** are two and three consequent words that frequently co-occur together.

**Exclude terms in > 99% and < 1% of documents** is necessary to remove words that are contentless words in the documents. This allows to reduce model noise.


**Corpus** is our collection of documents (i.e., our textual questionnaire responses) and <br>
**Dictionary** is a list of unique words in the corpus. It takes each unique word in the corpus and assigns them an index.

In [52]:
## Convert the text to lowercase
GI_Author_merged = GI_Author_merged.str.lower()

print("=======Full Dataset==============\n")
print(GI_Author_merged.head(1))

=======Full Dataset==============

0    get readers expose sexual violence report in a...
Name: GI_Author_merged, dtype: object


In [53]:
## Remove special characters
GI_Author_merged_regex = [re.sub(r'\'', '', sent) for sent in GI_Author_merged]
GI_Author_merged_regex = [re.sub(r'_', ' ',  sent) for sent in GI_Author_merged_regex]
GI_Author_merged_regex = [re.sub(r'[^\w\s]', '', sent) for sent in GI_Author_merged_regex]

In [54]:
## Remove alphanumerics, numbers, words that appear in the corpus less than twice and extra spaces.
GI_Author_merged_regex = [re.sub(r'\d', '',  sent) for sent in GI_Author_merged_regex]
GI_Author_merged_regex = [re.sub(r'\W*\b\w{1,2}\b', '',  sent) for sent in GI_Author_merged_regex]
GI_Author_merged_regex = [re.sub(r'\S*@\S*\s?', '', sent) for sent in GI_Author_merged_regex]

print("=======Full Dataset==============\n")
print("\n[INFO] GI_Author_merged....................\n")
print(GI_Author_merged_regex[:2])

=======Full Dataset==============


[INFO] GI_Author_merged....................

['get readers expose sexual violence report unbiased manner get interesting story pass class raise awareness report the facts get engagement write about something people care about expose the guy', '         ']


In [55]:
## Remove all punctuation and tokenize texts

#define helpful function
def tokenize(sentences):
    for sentence in sentences:
        yield(simple_preprocess(str(sentence), deacc=True))  # deacc=True removes all punctuation

#tolkenize full data set
GI_Author_merged_tokens = list(tokenize(GI_Author_merged_regex))


print("\n[INFO] GI_Author_merged....................\n")
print(GI_Author_merged_tokens[:2])


[INFO] GI_Author_merged....................

[['get', 'readers', 'expose', 'sexual', 'violence', 'report', 'unbiased', 'manner', 'get', 'interesting', 'story', 'pass', 'class', 'raise', 'awareness', 'report', 'the', 'facts', 'get', 'engagement', 'write', 'about', 'something', 'people', 'care', 'about', 'expose', 'the', 'guy'], []]


In [56]:
## Lemmatize words, keeping only noun, adj, vb, adv

#define helpful function
def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV', 'SCONJ', 'PRON', 'PART', 'NOUN', 'INTJ', 'AUX', 'ADV', 'ADP', 'ADJ']):
    """https://spacy.io/api/annotation"""
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent)) 
        texts_out.append([token.lemma_ for token in doc])
    return texts_out

#initialise Spacy
nlp = spacy.load('en', disable=['parser', 'ner'])

#lemmatize and select only noun, adj, vb, adv
GI_Author_merged_lemma = lemmatization(GI_Author_merged_tokens, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV', 'SCONJ', 'PRON', 'PART', 'NOUN', 'INTJ', 'AUX', 'ADV', 'ADP', 'ADJ'])
print(str(len(GI_Author_merged_lemma)))
print(GI_Author_merged_lemma[:4])

813
[['get', 'reader', 'expose', 'sexual', 'violence', 'report', 'unbiased', 'manner', 'get', 'interesting', 'story', 'pass', 'class', 'raise', 'awareness', 'report', 'the', 'fact', 'get', 'engagement', 'write', 'about', 'something', 'people', 'care', 'about', 'expose', 'the', 'guy'], [], [], []]


In [57]:
## Prepare to remove stop words
nltk.download('stopwords')
stopwords = set(nltk.corpus.stopwords.words('english'))
newStopWords =[str(x.strip()) for x in open(stopwords_location,'r').read().split('\n')]
stopwords.update(newStopWords)
print(len(stopwords))

219


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/hannahstevens/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [58]:
## Remove stop words 

#define helpful function
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) if word not in stopwords] for doc in texts]

#remove stop words 
GI_Author_merged_stopwords = remove_stopwords(GI_Author_merged_lemma)

print("\n[INFO] GI_Author_merged....................\n")
print(GI_Author_merged_stopwords[:2])


[INFO] GI_Author_merged....................

[['get', 'reader', 'expose', 'sexual', 'violence', 'report', 'unbiased', 'manner', 'get', 'interesting', 'story', 'pass', 'class', 'raise', 'awareness', 'report', 'fact', 'get', 'engagement', 'write', 'something', 'people', 'care', 'expose', 'guy'], []]


In [59]:
## Select Trigrams and Bigrams
            
GI_Author_merged_bigram = Phrases(GI_Author_merged_stopwords, min_count=3, delimiter=b' ', threshold=1)
GI_Author_merged_trigram = Phrases(GI_Author_merged_bigram[GI_Author_merged_stopwords], threshold=1)

GI_Author_merged_bigram_mod = gensim.models.phrases.Phraser(GI_Author_merged_bigram)
GI_Author_merged_trigram_mod = gensim.models.phrases.Phraser(GI_Author_merged_trigram)

for idx in range(len(GI_Author_merged_stopwords)):
    for token in GI_Author_merged_trigram_mod[GI_Author_merged_bigram_mod[GI_Author_merged_stopwords[idx]]]:
        #print(token)
        if ' ' in token:
            GI_Author_merged_stopwords[idx].append(token)
print("\n[INFO] GI_Author_merged....................\n")
print(GI_Author_merged_stopwords[:2])


[INFO] GI_Author_merged....................

[['get', 'reader', 'expose', 'sexual', 'violence', 'report', 'unbiased', 'manner', 'get', 'interesting', 'story', 'pass', 'class', 'raise', 'awareness', 'report', 'fact', 'get', 'engagement', 'write', 'something', 'people', 'care', 'expose', 'guy', 'get reader', 'raise awareness', 'expose guy'], []]


In [60]:
## Generate Corpus
dictionary_GI_Author_merged = gensim.corpora.Dictionary(GI_Author_merged_stopwords)
dictionary_GI_Author_merged.filter_extremes(no_below=.01, no_above=0.99)

## Generate Dictionary
corpus_GI_Author_merged = [dictionary_GI_Author_merged.doc2bow(text) for text in GI_Author_merged_stopwords]

## Save Corpus and Dictionary on a local drive
pickle.dump(corpus_GI_Author_merged, open('./output/corpus_GI_Author_merged.pkl', 'wb'))
dictionary_GI_Author_merged.save('./output/dictionary_GI_Author_merged.gensim')

### 3. Model selection (Selecting the number of topics (k))
A Model is represented by model Hyperparameters that define prior distribution of the topics within each document, and prior distribution of the different words within each topic. These should be defined based on theoretical assumptions about how we think the topics are actually distributed amongst our data. LDA model from gensim library has the following Hyperparameters:
- **Beta** (referred to as 'eta' in gensim) = the [distribution of the] number of words per topic
- **Alpha** =  the [distribution of the] number of topics per document



Both alpha and eta can be set to ‘symmetric’, ‘asymmetric’, or ‘auto’, where:
- ‘auto’ = the model learns the best values for the hyperparameters as it is trained on more and more data (i.e., it learns an asymmetric prior from the corpus). See http://jonathan-huang.org/research/dirichlet/dirichlet.pdf for an overview             
- 'asymmetric' = uses a fixed, normalized asymmetric prior of 1.0 / k (number of topics)
- 'symmetric' = uses a distribution of 1 / k (number of topics)



In Bayesian statistics, we have to define the distributions (i.e., prior distributions) of unknown variables (e.g., ϕ and θ) before running the data analysis. These should be defined based on theoretical assumptions about how we think the topics are actually distributed amongst our data. In our case, it makes sense to assume that some documents discuss more/less topics than other documents; thus, we set the document-topic distribution to be asymmetric. 


We recommend setting alpha = 'auto' as it sets the distribution to be asymmetric, and learns the best alpha value (i.e., lowest perplexity scores) from the data itself. It also makes sense to assume that some topics contain more words than others. Thus, we recommend setting the distribution of the number of words per topic to be asymmetric as well.

In addition, gensim LDA model has the following parameters:
- **Passes** = number of laps the model goes through the entire corpus (Increasing the number of passes reduces model bias)
- **Chunksize** = number of documents to load into memory at a time (smaller chunk sizes save memory, but take longer to train)
- **Update_every** = number of chunks to process before maximizing your model 
- **Random state** = sets the seed to make the model reproducible
- **Number of topics (k)**

**Number of topics (k)** defines the LDA model. Researchers must tell the model how many (k) prominent goal inference topics to sort each ‘bag of words’ document into. Problematically, several different k-values might work. Thus, we use a metric called perplexity to help us to determine the optimal number of topics. The utility in perplexity comes from comparing perplexity values across models with differing k-values to pinpoint the best model (i.e., the model with the lowest perplexity score). 

We recommend testing the perplexity of the model with a variety of k values, and then running the final model using the k-value with the selected perplexity score. **Model perplexity** is a frequently used metric that gauges how well a model fits the data.

In [61]:
## Set model Hyper Parameters
k = [1,2,3,4,5,6,7,8,9,10,11,12]
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta= 'auto'
per_word_topics=True

lda_model_GI_Author_merged = []

In [ ]:
## Get Perplexity Scores of Training Dataset
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

for i in k:
    lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                          id2word=dictionary_GI_Author_merged,
                                          num_topics=i, 
                                          random_state=random_state,
                                          update_every=update_every,
                                          chunksize=chunksize,
                                          passes=passes,
                                          iterations=iterations,
                                          alpha=alpha,
                                          eta=eta,                                                            
                                          per_word_topics=per_word_topics)

    print('\nPerplexity (num_topics = {}): '.format(i), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))


***********************************************************************
[INFO] GI_Author_Merged Full Dataset LDA Results....
***********************************************************************

Perplexity (num_topics = 1):  -6.19316089465453

Perplexity (num_topics = 2):  -6.253605099313884

Perplexity (num_topics = 3):  -6.294996413142641

Perplexity (num_topics = 4):  -6.332119621278861

Perplexity (num_topics = 5):  -6.3865077884785055

Perplexity (num_topics = 6):  -6.417232247865415


Usually, lower perplexity scores are indicative of increased model accuracy, and smaller k-values yield a more parsimonious set of topics. However, the perplexity score will often decrease as k increases. In these instances, it’s best to select the model that yields the highest perplexity value before the values flatten out. 

**Note: when selecting the optimal number of topics, we need to find a balance between overfitting and underfitting the model**

**Overfitting** (i.e., too many topics) can make it harder for human coders to label resulted topics since there is less coherence amongst the words in each topic. At the same time, resulting topics have less overlap in words.

**Underfitting** (i.e., too few topics) doesn't produce enough variance, limiting options for statistical analyses. Labeling resulting topics becomes easier since topics have more coherent list of words comprising each topic. At the same time, resulting topics have higher overlap in used words that leads to increased variance in the distribution of topics in each document.

**pyLDAvis visualization** of a model with selected value for k helps to asses Overfitting and Underfitting. Reading pyLDAvis:

Left pane:
- The area of each circle represents the prevalence of each topic over the entire corpus 
- The distance between the center of circles indicate the similarity between topics (i.e., inter-topic differences)

Right pane:
- If you hover over a particular topic on the left, the histogram on the right side lists the top 30 most relevant terms
- The widths of the gray bars represent the corpus-wide frequencies of each term, and the widths of the red bars represent the topic-specific frequencies of each term
- A slider at the top can adjust the relevance metric (λ); however, for our purposes, be sure it i set to λ = 1. For more information on the relevance metric, see library documentation. 

Documentation for this library can be found here: https://www.aclweb.org/anthology/W14-3110.pdf. 

**In the following steps we test LDA model hyper parameters for k in range [3-11].**

##### k = 3 topics

In [ ]:
## Initializing LDA Models and Parameters
topic_number = 3
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")


lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))

In [ ]:
## Analyze Model results
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 3...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)

##### k = 4 topics

In [ ]:
## Initializing LDA Models and Parameters
topic_number = 4
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))

In [ ]:
## Analyze Model results
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 4...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)

##### k = 5 topics

In [ ]:
## Initializing LDA Models and Parameters
topic_number = 5
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))

In [ ]:
## Analyze Model results
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 5...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)

###### Note:  with k=5 topics appear to be relatively spread out, with no overlapping topics.  At the same time, the the list of words comprising each topic appears coherent enough to label. 

##### k = 6 topics

In [ ]:
## Initializing LDA Models and Parameters
topic_number = 6
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))

In [ ]:
## Analyze Model results
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 6...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)

##### k = 7 topics

In [ ]:
## Initializing LDA Models and Parameters
topic_number = 7
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))

In [ ]:
## Analyze Model results
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 7...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)

##### k = 8 topics

In [ ]:
## Initializing LDA Models and Parameters
topic_number = 8
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))

In [ ]:
## Analyze Model results
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 8...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)

##### k = 9 topics

In [ ]:
## Initializing LDA Models and Parameters
topic_number = 9
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))

In [ ]:
## Analyze Model results
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 9...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)

##### k = 10 topics

In [ ]:
## Initializing LDA Models and Parameters
topic_number =10
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))

In [ ]:
## Analyze Model results
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 10...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)

##### k = 11 topics

In [ ]:
## Initializing LDA Models and Parameters
topic_number =11
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=850
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))

In [ ]:
## Analyze Model results
print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset Model Results....")
print("***********************************************************************")

print("\n[INFO] Num_topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("GI_Author_Merged.....k = 11...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)

### 4. Saving selected model results
As k=5 topics yielded the lowest perplexity value before the trend flattened out at k=6 topics and with k=5 topics appear to be relatively spread out, with no overlapping topics, we determined a k=5-topic LDA model would produce the simplest model of the models that explain the data well


To save results from the LDA model with selected parameters and number of topics we 
- rerun the model with k=5
- generate a column that tells us which topic each response contributed the most to
- save the analysis results to an excel file for topic validation

In [ ]:
## Initializing LDA Models and Parameters
topic_number = 5
random_state=42
update_every=1
chunksize=1800
passes=300
iterations=800
alpha='auto'
eta='auto'
per_word_topics=True

print("\n***********************************************************************")
print("[INFO] GI_Author_Merged Full Dataset LDA Results....")
print("***********************************************************************")

lda_model_GI_Author_merged = LdaModel(corpus=corpus_GI_Author_merged,
                                      id2word=dictionary_GI_Author_merged,
                                      num_topics=topic_number, 
                                      random_state=random_state,
                                      update_every=update_every,
                                      chunksize=chunksize,
                                      passes=passes,
                                      iterations=iterations,
                                      alpha=alpha,
                                      eta=eta,
                                      per_word_topics=per_word_topics)

print('\nPerplexity (topic_number = {}): '.format(topic_number), lda_model_GI_Author_merged.log_perplexity(corpus_GI_Author_merged))

In [ ]:
## GI_Author_Merged Model Results
print("\n***********************************************************************")
print("[INFO] Goal Inferences Model Results....")
print("***********************************************************************")

print("\n[INFO] Number of topics: {}\n".format(topic_number))
topics = lda_model_GI_Author_merged.show_topics(num_topics=topic_number, num_words=10, log=True, formatted=True)
for topic in topics:
    print(topic)

print("Goal Inferences .....k = 5...................")
lda_display = pyLDAvis.gensim.prepare(lda_model_GI_Author_merged, corpus_GI_Author_merged, dictionary_GI_Author_merged)
pyLDAvis.display(lda_display)

In [ ]:
## Generate a column that tells us which topic each response contributed the most to

#define helpful function
def format_topics_sentences(ldamodel, corpus, texts):
    # Init output
    sent_topics_df = pd.DataFrame()

    # Get the main topic of each document
    for i, row_list in enumerate(ldamodel[corpus]):
        row = row_list[0] if ldamodel.per_word_topics else row_list            
        row = sorted(row, key=lambda x: (x[1]), reverse=True)
        
        # Get the Dominant topic, Perc Contribution, and Keywords for each document
        raw_frame = {}
        for j, (topic_num, prop_topic) in enumerate(row):
            if j==0:
                raw_frame['Dominant'] = topic_num

            raw_frame['Topic' + str(topic_num)] = round(prop_topic, 4)
            
        df = pd.DataFrame(data=raw_frame, index=[0])
        sent_topics_df = sent_topics_df.append(df)

    return(sent_topics_df)


df_topic_sents_keywords_GI_Author_merged = format_topics_sentences(ldamodel=lda_model_GI_Author_merged, 
                                                                   corpus=corpus_GI_Author_merged, 
                                                                   texts=GI_Author_merged_stopwords)

#rename index of the dataframe
df_dominant_topic_GI_Author_merged = df_topic_sents_keywords_GI_Author_merged.reset_index()
df_dominant_topic_GI_Author_merged.index.name='Document_No';

print(df_dominant_topic_GI_Author_merged.head(812))

In [ ]:
## Generate a data frame to export the results into
lda_topics_GI_Author_merged = np.array(df_dominant_topic_GI_Author_merged['Dominant'])
topic0_contrib_lda_topics_GI_Author_merged = np.array(df_dominant_topic_GI_Author_merged['Topic0'])
topic1_contrib_lda_topics_GI_Author_merged = np.array(df_dominant_topic_GI_Author_merged['Topic1'])
topic2_contrib_lda_topics_GI_Author_merged = np.array(df_dominant_topic_GI_Author_merged['Topic2'])
topic3_contrib_lda_topics_GI_Author_merged = np.array(df_dominant_topic_GI_Author_merged['Topic3'])
topic4_contrib_lda_topics_GI_Author_merged = np.array(df_dominant_topic_GI_Author_merged['Topic4'])

GI_Author_merged = np.array(data['GI_Author_merged'])

ResponseId = np.array(data['ResponseId'])
uid = np.array(data['uid'])

results = { 
    'uid' : uid,'ResponseId': ResponseId, 
    'GI_Author_merged': GI_Author_merged, 
    'lda_topics_GI_Author_merged': lda_topics_GI_Author_merged, 
    'topic0_contrib_lda_topics_GI_Author_merged':topic0_contrib_lda_topics_GI_Author_merged,
    'topic1_contrib_lda_topics_GI_Author_merged':topic1_contrib_lda_topics_GI_Author_merged,
    'topic2_contrib_lda_topics_GI_Author_merged':topic2_contrib_lda_topics_GI_Author_merged,
    'topic3_contrib_lda_topics_GI_Author_merged':topic3_contrib_lda_topics_GI_Author_merged,
    'topic4_contrib_lda_topics_GI_Author_merged':topic4_contrib_lda_topics_GI_Author_merged,

}

frame = pd.DataFrame(results, columns = [
                                        'uid', 'ResponseId',
                                        'GI_Author_merged', 'lda_topics_GI_Author_merged', 
                                        'topic0_contrib_lda_topics_GI_Author_merged',
                                        'topic1_contrib_lda_topics_GI_Author_merged',
                                        'topic2_contrib_lda_topics_GI_Author_merged',
                                        'topic3_contrib_lda_topics_GI_Author_merged',
                                        'topic4_contrib_lda_topics_GI_Author_merged',
                                        ])

In [ ]:
## Export results to an .xlsx file
frame.to_excel("./output/lda_results_full_dataset_topic_num_5.xlsx")
frame.to_excel("./output/lda_results_All_data_topic_num_5.xlsx")